1. Train a simple neural network to classify whether a review is positive or negative using a small dataset (you can use any dataset of your choice), then save the trained model to a .h5 file using model.save('sentiment_model.h5').

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TextVectorization, Embedding, GlobalAveragePooling1D, Dense

reviews = [
    "This movie is amazing",
    "I loved this film",
    "Fantastic acting",
    "Very good experience",
    "I hate this movie",
    "Worst film ever",
    "Terrible acting",
    "Very boring movie"
]

labels = [1, 1, 1, 1, 0, 0, 0, 0]

vectorizer = TextVectorization(max_tokens=1000, output_mode="int", output_sequence_length=20)
vectorizer.adapt(reviews)

x_train = vectorizer(tf.constant(reviews))

model = Sequential([
    Embedding(1000, 16),
    GlobalAveragePooling1D(),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(x_train, labels, epochs=20)

model.save("sentiment_model.h5")

2. Write code to load the 'sentiment_model.h5' file you saved and use it to predict the sentiment of three new sample reviews.

In [ ]:
import tensorflow as tf
import numpy as np

loaded_model = tf.keras.models.load_model("sentiment_model.h5")

new_reviews = [
    "Excellent movie",
    "Very bad film",
    "I really enjoyed it"
]

x_new = vectorizer(tf.constant(new_reviews))

predictions = loaded_model.predict(x_new)

for review, prediction in zip(new_reviews, predictions):
    print(review, "->", prediction[0])

3. Implement model checkpointing during training so that the model's weights are saved every time the validation accuracy improves.<br><br><em><strong>Hint:</strong> Use the ModelCheckpoint callback in Keras with save_best_only=True.</em>

In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

model.fit(
    x_train,
    labels,
    epochs=20,
    validation_split=0.2,
    callbacks=[checkpoint]
)

4. Export your trained model in a format suitable for deployment (such as TensorFlow SavedModel format), and explain in one line how this format helps with deploying the model to a web or mobile app.

In [ ]:
model.export("saved_model")

5. Use ChatGPT or Copilot to help you write the code for saving both the model architecture and weights separately, then test loading them back and confirm the loaded model gives the same predictions as the original.

In [ ]:
import tensorflow as tf

model_json = model.to_json()

with open("model.json", "w") as file:
    file.write(model_json)

model.save_weights("model.weights.h5")

with open("model.json", "r") as file:
    loaded_json = file.read()

loaded_model = tf.keras.models.model_from_json(loaded_json)

loaded_model.load_weights("model.weights.h5")

loaded_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

pred1 = model.predict(x_new)
pred2 = loaded_model.predict(x_new)

print("Original Model Predictions")
print(pred1)

print("Loaded Model Predictions")
print(pred2)

print("Same Predictions:", np.allclose(pred1, pred2))